# Reversed distillation (warm-start): Transformer teacher -> skeleton student

The forward experiments distil from the skeleton model into the Signformer. Here the
roles swap: the **Signformer is the teacher** (39.54 dev WER) and the **skeleton model
is the student** (47.81). This is the conventional setup a stronger model teaching a
weaker one which the thesis names as the experiment its weak-teacher result leaves
untested.

This run trains the skeleton student **warm-start**. The teacher never trains: its
encoder features were exported once by `extract_transformer_feats.py`.

**How to read it.** The loop here is single-phase, not the two-phase schedule that
produced the 47.81 baseline, so do not compare against 47.81 directly. Instead run the
matched **control** arm (`RUN_CONTROL = True`): identical loop, distillation switched
off. The difference between the distilled arm and the control isolates the teacher, the
same discipline the forward controlled experiment uses.

In [ ]:
# Run from code/distillation/ no matter where the kernel was launched.
import os
if not os.path.exists('fd_cmkd.py'):
    _cwd = os.getcwd()
    for _k in range(0, 6):
        _base = os.path.abspath(os.path.join(_cwd, *(['..'] * _k)))
        _cand = os.path.join(_base, 'code', 'distillation')
        if os.path.exists(os.path.join(_cand, 'fd_cmkd.py')):
            os.chdir(_cand); break
        if os.path.exists(os.path.join(_base, 'fd_cmkd.py')):
            os.chdir(_base); break
print('working dir:', os.getcwd())

# Where this run writes. Output goes to runs/<RUN_NAME>/; nothing else is touched.
import os

RUN_NAME    = 'reverse_warm'
OVERWRITE   = False
DO_TRAIN    = False    # True runs the training below
RUN_CONTROL = True     # also train the matched no-teacher control arm

RUN_DIR = os.path.join('../../runs', RUN_NAME)
if os.path.exists(RUN_DIR) and not OVERWRITE:
    _n = 2
    while os.path.exists('%s_%03d' % (RUN_DIR, _n)):
        _n += 1
    RUN_DIR = '%s_%03d' % (RUN_DIR, _n)
print('this run writes to:', os.path.normpath(os.path.abspath(RUN_DIR)))

In [ ]:
import sys

NB_DIR = os.path.abspath('.')
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

from reverse_distill import train_reverse
import reverse_distill as rd
print('imports OK  | device:', rd.DEVICE)

In [ ]:
# The teacher features must exist. Produce them once with:
#     python extract_transformer_feats.py
tf_dir = os.path.join('../../dataset/features/transformer_feats')
tf_train = os.path.join(tf_dir, 'train.pkl')
assert os.path.exists(tf_train), (
    'missing transformer teacher features: ' + tf_train +
    '  -- run: python extract_transformer_feats.py')
import pickle
_t = pickle.load(open(tf_train, 'rb'))
print('teacher features   : %d videos, dim %d' % (len(_t), next(iter(_t.values())).shape[1]))
print('regime             : warm-start')

In [ ]:
# Knobs. WARM_START fixes the regime for this notebook; leave it as is.
WARM_START   = True
LAMBDA_FEAT  = 0.5      # weight of the FD-CMKD feature loss
LOW_W        = 1.0      # low-frequency band (strong consistency)
HIGH_W       = 0.25     # high-frequency band (weak consistency)
WARMUP_STEPS = 500      # ramp the distillation weight over the first N steps
EPOCHS       = None     # None -> skeleton config default (75)
SUBSET       = None     # e.g. 96 for a quick smoke test; None for the full set
SEED         = 42

In [ ]:
# Train the distilled arm, and optionally the matched control (no teacher).
results = {}
if DO_TRAIN:
    print('=== distilled arm (Transformer teacher -> skeleton student) ===')
    wer_d, ck_d = train_reverse(
        run_dir=os.path.join(RUN_DIR, 'distill'), warm_start=WARM_START,
        enabled=True, lambda_feat=LAMBDA_FEAT, low_w=LOW_W, high_w=HIGH_W,
        distill_warmup_steps=WARMUP_STEPS, epochs=EPOCHS, subset=SUBSET, seed=SEED)
    results['distill'] = wer_d
    if RUN_CONTROL:
        print('=== control arm (identical loop, no teacher) ===')
        wer_c, ck_c = train_reverse(
            run_dir=os.path.join(RUN_DIR, 'control'), warm_start=WARM_START,
            enabled=False, epochs=EPOCHS, subset=SUBSET, seed=SEED)
        results['control'] = wer_c
else:
    print('DO_TRAIN = False -> skipping training. Set it True to run.')

In [ ]:
# Curves for both arms: train and validation loss (left), dev WER (right).
import matplotlib.pyplot as plt

def _read(vf):
    ep, tr, vl, wr = [], [], [], []
    if os.path.exists(vf):
        for line in open(vf):
            t = line.replace(':', ' ').split()
            try:
                ep.append(int(t[t.index('epoch') + 1]))
                tr.append(float(t[t.index('trainloss') + 1]))
                vl.append(float(t[t.index('valloss') + 1]))
                wr.append(float(t[t.index('WER') + 1]))
            except (ValueError, IndexError):
                pass
    return ep, tr, vl, wr

arms = {'distill': os.path.join(RUN_DIR, 'distill', 'validations.txt'),
        'control': os.path.join(RUN_DIR, 'control', 'validations.txt')}
colors = {'distill': 'tab:red', 'control': 'tab:blue'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
any_data = False
for name, vf in arms.items():
    ep, tr, vl, wr = _read(vf)
    if not ep:
        continue
    any_data = True
    c = colors[name]
    ax1.plot(ep, tr, lw=0.9, alpha=0.5, color=c, label='%s train' % name)
    ax1.plot(ep, vl, marker='o', ms=3, color=c, label='%s val' % name)
    ax2.plot(ep, wr, marker='o', ms=3, color=c, label=name)

if not any_data:
    print('no validations yet -- train first (DO_TRAIN = True).')
else:
    ax1.set_xlabel('epoch'); ax1.set_ylabel('CTC loss'); ax1.set_title('Loss')
    ax1.legend(); ax1.grid(alpha=0.3)
    ax2.set_xlabel('epoch'); ax2.set_ylabel('dev WER (%)'); ax2.set_title('Dev WER')
    ax2.legend(); ax2.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(RUN_DIR, 'curves.png'), dpi=120)
    plt.show()

In [ ]:
# Verdict. The comparison that matters is distilled vs its matched control.
if results:
    print('Reverse direction (%s):' % ('warm-start' if WARM_START else 'from scratch'))
    for k in ('control', 'distill'):
        if k in results:
            print('  %-8s dev WER  %.2f' % (k, results[k]))
    if 'control' in results and 'distill' in results:
        delta = results['distill'] - results['control']
        verdict = ('distillation HELPS' if delta < -0.30 else
                   'distillation HURTS' if delta > 0.30 else 'NEUTRAL')
        print('  delta (distill - control): %+.2f  ->  %s (margin 0.30)' % (delta, verdict))
else:
    print('no results yet -- train first (DO_TRAIN = True).')